# client-simple: An OpenAI model using an MCP tool

**This client connects to the knowledge base server over stdio, hands the server's tool to an OpenAI model, and runs the tool whenever the model asks for it.** It's the function-calling loop from `3-tools.py`, with one difference: The tool runs in the MCP server, not in this notebook.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 MCP server           this notebook (host)       OpenAI model
     |                       |                         |
     |&lt;--- 1. tools/list ----|                         |
     |---- the tool --------&gt;|                         |
     |                       |-- 2. question+tools ---&gt;|
     |                       |&lt;-- call the tool -------|
     |&lt;--- 3. tools/call ----|                         |
     |---- Q&amp;A text --------&gt;|                         |
     |                       |-- 4. result as "tool" -&gt;|
     |                       |&lt;---- final answer ------|
</pre>

Read first: Section 8 of `../1-understanding-mcp/README.md`, the notebooks in `../2-simple-server-setup`, and `server.ipynb` here.

## 1. Setup

- **`load_dotenv()`:** Reads `OPENAI_API_KEY` from a `.env` file. It searches this folder and the folders above it, and finds `crash-course/.env`.
- **`AsyncOpenAI()`:** The same OpenAI client as `OpenAI()`, except that its calls are awaited. That suits async code like an MCP client: While it waits for OpenAI, the program can keep the MCP connection running.
- **`server`:** The command that starts `server.py` over stdio, as in Part 2.

In [1]:
import json

from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import AsyncOpenAI

load_dotenv()

openai_client = AsyncOpenAI()
model = "gpt-4o-mini"
server = StdioServerParameters(command="python", args=["server.py"])

## 2. Turning MCP tools into OpenAI tools

MCP and OpenAI describe a tool with the same three facts, under slightly different names. MCP gives `name`, `description` and `inputSchema`. OpenAI wants `{"type": "function", "function": {...}}` with `name`, `description` and `parameters`. So `get_mcp_tools` lists the server's tools and rewraps each one, putting `inputSchema` under `parameters`.

In [2]:
async def get_mcp_tools(session):
    tools_result = await session.list_tools()

    tools = []
    for tool in tools_result.tools:
        tools.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema,
                },
            }
        )

    return tools

## 3. Answering a question

`process_query` is the function-calling loop:

1. **Get the tools** from the server, in OpenAI's format.
2. **First call:** Send the question and the tools to the model. By default, the model decides whether it needs a tool.
3. **If the model asked for tools:** For each tool call, read its `name` and `args`, run it on the MCP server with `session.call_tool`, and add the result to `messages` as a `"tool"` message. The arguments arrive as a JSON string, so `json.loads` turns them into a dict, as in `3-tools.py`.
4. **Second call:** Send everything back. `tool_choice="none"` stops the model from asking for another tool, so it has to write the answer.
5. **If the model didn't ask for a tool:** Its first reply already is the answer.

In [3]:
async def process_query(session, query):
    tools = await get_mcp_tools(session)
    messages = [{"role": "user", "content": query}]

    response = await openai_client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
    )
    assistant_message = response.choices[0].message
    messages.append(assistant_message)

    if assistant_message.tool_calls:
        for tool_call in assistant_message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print("The model called:", name, args)

            result = await session.call_tool(name, args)
            messages.append(
                {"role": "tool", "tool_call_id": tool_call.id, "content": result.content[0].text}
            )

        final_response = await openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="none",
        )
        return final_response.choices[0].message.content
    else:
        return assistant_message.content

## 4. Running it

The connection is the one from Part 2: `stdio_client` starts the server, `ClientSession` speaks MCP, and `errlog=log` sends the server's stderr to `server_log.txt`. Inside it, the cell saves the tool list so section 5 can show it, then asks the question.

In [4]:
query = "What is our company's vacation policy?"

with open("server_log.txt", "w") as log:
    async with stdio_client(server, errlog=log) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await get_mcp_tools(session)
            answer = await process_query(session, query)

print(answer)

The model called: get_knowledge_base {}


Our company's vacation policy states that full-time employees are entitled to 20 paid vacation days per year. You can take vacation days after completing 6 months of employment. Unused vacation days can be carried over to the next year, up to a maximum of 5 days. Vacation requests should be submitted at least 2 weeks in advance through the HR portal.


The model chose `get_knowledge_base` by itself, with no arguments (`{}`), and then answered from the text the tool returned.

## 5. What the model was given

This is the tool list in OpenAI's format, exactly as it went to the model. `json.dumps(tools, indent=2)` prints it as indented JSON.

In [5]:
print(json.dumps(tools, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "get_knowledge_base",
      "description": "Retrieve the company knowledge base, with every question and answer about company policies.",
      "parameters": {
        "properties": {},
        "title": "get_knowledge_baseArguments",
        "type": "object"
      }
    }
  }
]


## 6. What the server logged

In [6]:
with open("server_log.txt") as log:
    print(log.read())

Processing request of type ListToolsRequest
Processing request of type ListToolsRequest
Processing request of type CallToolRequest



There are two `tools/list` requests: One from the cell above, so section 5 could print the tools, and one inside `process_query`. There's one `tools/call`: The model's single tool call.

## 7. The .py version

`client-simple.py` has the same setup and functions. Only the running part differs, because a plain file needs `async def main()` and `asyncio.run(main())`:

<div style="font-size: 0.85em">

```python
async def main():
    with open("server_log.txt", "w") as log:
        async with stdio_client(server, errlog=log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                query = "What is our company's vacation policy?"
                print(query)
                answer = await process_query(session, query)
                print(answer)


asyncio.run(main())
```

</div>

Run from this folder with `python client-simple.py`, it printed:

<div style="font-size: 0.85em">

```text
What is our company's vacation policy?
The model called: get_knowledge_base {}
Our company's vacation policy is as follows:

- Full-time employees are entitled to 20 paid vacation days per year.
- Vacation days can be taken after completing 6 months of employment.
- Unused vacation days can be carried over to the next year, up to a maximum of 5 days.
- Vacation requests must be submitted at least 2 weeks in advance through the HR portal.
```

</div>

The model writes its answer fresh each time, so the wording changes from run to run.

## Say it out loud

> "My host connects to the MCP server over stdio, lists its tools and converts each one into OpenAI's tool format: The name, the description, and the input schema as `parameters`. Then it runs the normal function-calling loop. The model decides whether to call a tool, my code executes the call on the MCP server with `call_tool`, sends the result back as a `tool` message, and the model writes the final answer. The model never talks to the MCP server. My code is the bridge."